<a href="https://colab.research.google.com/github/Codewithtanvi/House-Price-Prediction-/blob/main/Day_2_Data_Cleaning_and_SQL_Database_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import sqlite3

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/Bluestock Internship/Datasets/drive-download-20260603T142152Z-3-001.zip'
extraction_path = '/content/extracted_data' # Choose a directory to extract to
output_directory = '/content/data/processed' # Define the output directory

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

# Now, read the CSV from the extracted location
nav = pd.read_csv(f'{extraction_path}/02_nav_history.csv')

# Convert date
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Sort
nav = nav.sort_values(['amfi_code','date'])

# Remove duplicates
nav = nav.drop_duplicates()

# Forward fill NAV for each scheme
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

# Validate NAV
nav = nav[nav['nav'] > 0]

print(nav.info())
print(nav.head())

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

nav.to_csv(f'{output_directory}/nav_history_cleaned.csv',
           index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB
None
      amfi_code       date       nav
5750     100016 2022-01-03  520.4608
5751     100016 2022-01-04  515.0971
5752     100016 2022-01-05  521.7239
5753     100016 2022-01-06  515.7880
5754     100016 2022-01-07  515.1639


In [5]:
import os

# List the contents of your Google Drive
print('Listing contents of /content/drive/MyDrive/')
print(os.listdir('/content/drive/MyDrive'))

# You might need to explore further if your file is in a subdirectory
# For example, to list a specific folder:
# print('\nListing contents of /content/drive/MyDrive/data/raw/')
# print(os.listdir('/content/drive/MyDrive/data/raw'))

Listing contents of /content/drive/MyDrive/
['Colab Notebooks', 'project_2.py', 'LOGO.png', 'wetransfer_project-file_2024-05-29_0929.zip', 'Types Of Dals Collection', 'images of number plate', 'SUPERMARKET GROCERY SALES- RETAIL ANALYSIS.pptx', 'Laptop Price Analysis.pptx', 'Resume (2) (2).pdf', 'Resume (2) (1).pdf', 'Documents', 'Certificates', 'Research Paper', 'Aptitute by Raman Tiwari.docx', 'Cognifz offer letter.pdf', 'Task Completed (1).zip', 'Task Completed.zip', 'Fees Receipt (1).jpg', 'Fees Receipt.jpg', 'Screenshot_20250731_110150_YouTube.jpg', 'Screenshot_20250820_123613_PhonePe.jpg', 'Resume (2).pdf', 'CV (6).pdf', 'CV (5).pdf', 'Document from Tanii.pdf', 'CV (4).pdf', 'CV (3).pdf', 'CV (2).pdf', 'CV (1).pdf', 'CV.pdf', 'urbanthreads_q4.csv', 'urbanthreads_q4 (2).gsheet', 'urbanthreads_q4 (1).gsheet', 'urbanthreads_q4.gsheet', 'Raw_data.gsheet', 'Cleaning_Log.gsheet', 'RBI+GRADE+B+PYQ+(1).pdf', 'Bluestock Internship']


In [6]:
txn = pd.read_csv('/content/extracted_data/08_investor_transactions.csv')

txn['transaction_type'] = (
    txn['transaction_type']
    .str.strip()
    .str.upper()
)

mapping = {
    'SIP':'SIP',
    'SYSTEMATIC INVESTMENT PLAN':'SIP',
    'LUMPSUM':'Lumpsum',
    'LUMP SUM':'Lumpsum',
    'REDEMPTION':'Redemption'
}

txn['transaction_type'] = txn['transaction_type'].replace(mapping)

In [7]:
txn['transaction_date'] = pd.to_datetime(
    txn['transaction_date'],
    errors='coerce'
)

In [8]:
invalid_amounts = txn[txn['amount_inr'] <= 0]

print("Invalid Amount Records:")
print(invalid_amounts)

txn = txn[txn['amount_inr'] > 0]

Invalid Amount Records:
Empty DataFrame
Columns: [investor_id, transaction_date, amfi_code, transaction_type, amount_inr, state, city, city_tier, age_group, gender, annual_income_lakh, payment_mode, kyc_status]
Index: []


In [9]:
valid_kyc = [
    'Verified',
    'Pending',
    'Rejected'
]

invalid_kyc = txn[
    ~txn['kyc_status'].isin(valid_kyc)
]

print(invalid_kyc)

Empty DataFrame
Columns: [investor_id, transaction_date, amfi_code, transaction_type, amount_inr, state, city, city_tier, age_group, gender, annual_income_lakh, payment_mode, kyc_status]
Index: []


In [10]:
txn.to_csv(
    '/content/data/processed/investor_transactions_cleaned.csv',
    index=False
)

In [11]:
perf = pd.read_csv('/content/extracted_data/07_scheme_performance.csv')

In [12]:
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct'
]

for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors='coerce'
    )

In [13]:
anomalies = perf[
    (perf['return_1yr_pct'] > 100) |
    (perf['return_1yr_pct'] < -100)
]

print(anomalies)

Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [14]:
invalid_expense = perf[
    (perf['expense_ratio_pct'] < 0.1) |
    (perf['expense_ratio_pct'] > 2.5)
]

print(invalid_expense)

Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [15]:
perf.to_csv(
    '/content/data/processed/scheme_performance_cleaned.csv',
    index=False
)

SQLite

In [16]:
conn = sqlite3.connect(':memory:') # Use an in-memory SQLite database
cursor = conn.cursor()

# Define the SQL statements
sql_statements = [
    """CREATE TABLE dim_fund (
        fund_id INTEGER PRIMARY KEY,
        amfi_code INTEGER UNIQUE,
        fund_name TEXT,
        category TEXT,
        fund_house TEXT
    );""",

    """CREATE TABLE dim_date (
        date_id INTEGER PRIMARY KEY,
        full_date DATE,
        day INTEGER,
        month INTEGER,
        quarter INTEGER,
        year INTEGER
    );""",

    """CREATE TABLE fact_nav (
        nav_id INTEGER PRIMARY KEY,
        fund_id INTEGER,
        date_id INTEGER,
        nav REAL,
        FOREIGN KEY (fund_id)
            REFERENCES dim_fund(fund_id),
        FOREIGN KEY (date_id)
            REFERENCES dim_date(date_id)
    );""",

    """CREATE TABLE fact_transactions (
        transaction_id INTEGER PRIMARY KEY,
        fund_id INTEGER,
        date_id INTEGER,
        amount REAL,
        transaction_type TEXT,
        FOREIGN KEY (fund_id)
            REFERENCES dim_fund(fund_id),
        FOREIGN KEY (date_id)
            REFERENCES dim_date(date_id)
    );""",

    """CREATE TABLE fact_performance (
        performance_id INTEGER PRIMARY KEY,
        fund_id INTEGER,
        return_1y REAL,
        return_3y REAL,
        return_5y REAL,
        expense_ratio REAL,
        FOREIGN KEY (fund_id)
            REFERENCES dim_fund(fund_id)
    );""",

    """CREATE TABLE fact_aum (
        aum_id INTEGER PRIMARY KEY,
        fund_id INTEGER,
        date_id INTEGER,
        aum REAL,
        FOREIGN KEY (fund_id)
            REFERENCES dim_fund(fund_id),
        FOREIGN KEY (date_id)
            REFERENCES dim_date(date_id)
    );"""
]

# Execute each statement
for sql in sql_statements:
    cursor.execute(sql)

print("All tables created successfully in the in-memory SQLite database.")

All tables created successfully in the in-memory SQLite database.


In [17]:
engine = create_engine('sqlite://') # Let SQLAlchemy manage its own in-memory database

In [18]:
nav.to_sql(
    'nav_history',
    engine,
    if_exists='replace',
    index=False
)

txn.to_sql(
    'investor_transactions',
    engine,
    if_exists='replace',
    index=False
)

perf.to_sql(
    'scheme_performance',
    engine,
    if_exists='replace',
    index=False
)

40

In [19]:
print("NAV:", len(nav))
print("TXN:", len(txn))
print("PERF:", len(perf))

with engine.connect() as conn:
    print(
        pd.read_sql(
            "SELECT COUNT(*) c FROM nav_history",
            conn
        )
    )

NAV: 46000
TXN: 32778
PERF: 40
       c
0  46000


In [20]:
# Populate dim_date table
# Extract unique dates from nav, txn, and perf dataframes
all_dates = pd.concat([
    nav['date'],
    txn['transaction_date']
]).unique()

dates_df = pd.DataFrame({'full_date': all_dates})
dates_df['full_date'] = pd.to_datetime(dates_df['full_date'])

dates_df['day'] = dates_df['full_date'].dt.day
dates_df['month'] = dates_df['full_date'].dt.month
dates_df['quarter'] = dates_df['full_date'].dt.quarter
dates_df['year'] = dates_df['full_date'].dt.year

# Sort dates
dates_df = dates_df.sort_values('full_date').reset_index(drop=True)

# Insert into dim_date table
dates_df.to_sql(
    'dim_date',
    engine,
    if_exists='replace',
    index_label='date_id'
)

print("Data loaded into dim_date table successfully.")

Data loaded into dim_date table successfully.


In [21]:
# Load date_id and full_date from the dim_date table in the database
db_dim_date_map = pd.read_sql("SELECT date_id, full_date FROM dim_date", engine)
db_dim_date_map['full_date'] = pd.to_datetime(db_dim_date_map['full_date'])
print("Date ID mapping loaded from database:")
print(db_dim_date_map.head())

Date ID mapping loaded from database:
   date_id  full_date
0        0 2022-01-03
1        1 2022-01-04
2        2 2022-01-05
3        3 2022-01-06
4        4 2022-01-07


In [24]:
import os

print(os.path.exists("bluestock_mf.db"))


False


In [25]:
!find /content -name "*.db"

/content/.config/default_configs.db
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db


In [26]:
import sqlite3

conn = sqlite3.connect('/content/bluestock_mf.db')

In [27]:
import pandas as pd

query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name


In [28]:
pd.read_sql(
    "PRAGMA table_info(nav_history);",
    conn
)

,cid,name,type,notnull,dflt_value,pk


In [30]:
query = """
SELECT *
FROM nav_history
LIMIT 5;
"""

pd.read_sql(query, engine)

,amfi_code,date,nav
0,100016,2022-01-03 00:00:00.000000,520.4608
1,100016,2022-01-04 00:00:00.000000,515.0971
2,100016,2022-01-05 00:00:00.000000,521.7239
3,100016,2022-01-06 00:00:00.000000,515.7880
4,100016,2022-01-07 00:00:00.000000,515.1639


In [33]:
import pandas as pd

# Ensure nav_history is loaded into the engine's database
# This is added as a robust fix for persistent 'no such table' errors
# in in-memory SQLite, especially if the engine state is intermittently lost.
# The 'nav' DataFrame must be available in the global scope for this to work.
nav.to_sql(
    'nav_history',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql(
    "SELECT * FROM nav_history LIMIT 5",
    engine
)

,amfi_code,date,nav
0,100016,2022-01-03 00:00:00.000000,520.4608
1,100016,2022-01-04 00:00:00.000000,515.0971
2,100016,2022-01-05 00:00:00.000000,521.7239
3,100016,2022-01-06 00:00:00.000000,515.7880
4,100016,2022-01-07 00:00:00.000000,515.1639


In [34]:
try:
    pd.read_sql(query, conn)
except Exception as e:
    print(e)

In [35]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,nav_history


In [36]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///bluestock_mf.db')

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", engine)

,name
0,nav_history


In [37]:
pd.read_sql("""
SELECT *
FROM nav_history
LIMIT 5;
""", engine)


,amfi_code,date,nav
0,100016,2022-01-03 00:00:00.000000,520.4608
1,100016,2022-01-04 00:00:00.000000,515.0971
2,100016,2022-01-05 00:00:00.000000,521.7239
3,100016,2022-01-06 00:00:00.000000,515.7880
4,100016,2022-01-07 00:00:00.000000,515.1639


In [38]:
with engine.begin() as conn:
    conn.exec_driver_sql("""
    CREATE TABLE fact_aum (
        aum_id INTEGER PRIMARY KEY,
        fund_id INTEGER,
        aum REAL
    );
    """)

In [41]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", engine)

,name
0,nav_history
1,fact_aum


In [45]:
import pandas as pd
from sqlalchemy import text

# --- Setup: Ensure dim_fund and fact_aum tables are created and populated ---
# This is added to ensure the query has the necessary tables and data,
# as their setup might have been inconsistent due to previous cell deletions.

# 1. Create dim_fund table if not exists and populate it
# Extract unique fund information from 'perf' DataFrame
db_dim_fund_map = None # Initialize to None
try:
    # Try to read existing dim_fund to avoid re-creating fund_id mappings if it already exists from another cell run
    db_dim_fund_map = pd.read_sql(text("SELECT fund_id, amfi_code FROM dim_fund"), engine)
except Exception: # If table doesn't exist, this will fail
    pass # Proceed to create and populate

if db_dim_fund_map is None or db_dim_fund_map.empty:
    dim_fund_df = perf[['amfi_code', 'scheme_name', 'category', 'fund_house']].drop_duplicates().copy()
    dim_fund_df = dim_fund_df.rename(columns={'scheme_name': 'fund_name'})
    dim_fund_df.to_sql(
        'dim_fund',
        engine,
        if_exists='replace',
        index_label='fund_id'
    )
    # Reload map after creation
    db_dim_fund_map = pd.read_sql(text("SELECT fund_id, amfi_code FROM dim_fund"), engine)

# 2. Populate fact_aum table
# Assuming 'perf' DataFrame contains 'aum_crore' and 'amfi_code'
fact_aum_df_to_load = perf.merge(db_dim_fund_map, on='amfi_code', how='left')
fact_aum_df_to_load = fact_aum_df_to_load[['fund_id', 'aum_crore']]
fact_aum_df_to_load = fact_aum_df_to_load.rename(columns={'aum_crore': 'aum'})
fact_aum_df_to_load.to_sql(
    'fact_aum',
    engine,
    if_exists='replace',
    index=False
)
# --- End Setup ---

# Corrected SQL query to get Top 5 Funds by Total AUM
query_aum = """
SELECT df.fund_name,
       SUM(fa.aum) AS total_aum
FROM fact_aum fa
JOIN dim_fund df ON fa.fund_id = df.fund_id
GROUP BY df.fund_name
ORDER BY total_aum DESC
LIMIT 5;
"""

print("\n--- Top 5 Funds by Total AUM ---")
with engine.connect() as conn:
    result_aum = pd.read_sql(text(query_aum), conn)
    print(result_aum)



--- Top 5 Funds by Total AUM ---
                                           fund_name  total_aum
0  Mirae Asset Emerging Bluechip Fund - Regular -...      49046
1      Kotak Emerging Equity Fund - Regular - Growth      47469
2     Nippon India Small Cap Fund - Regular - Growth      43630
3         DSP Top 100 Equity Fund - Regular - Growth      41828
4                UTI Mid Cap Fund - Regular - Growth      41728


In [51]:
import pandas as pd
from sqlalchemy import text

# Ensure the 'investor_transactions' table exists and is populated.
# This re-loads the 'txn' DataFrame into the engine's database.
txn.to_sql(
    'investor_transactions',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql(text("""
SELECT *
FROM investor_transactions
ORDER BY amount_inr DESC
LIMIT 10;
"""), engine)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003045,2025-05-10 00:00:00.000000,118632,Redemption,597498,Telangana,Hyderabad,T30,26-35,Male,12.2,Cheque,Verified
1,INV004871,2024-09-09 00:00:00.000000,119551,Lumpsum,594649,Punjab,Chandigarh,T30,26-35,Male,9.2,Net Banking,Verified
2,INV002689,2024-12-31 00:00:00.000000,101208,Lumpsum,593281,Madhya Pradesh,Bhopal,B30,26-35,Male,7.4,Cheque,Verified
3,INV004413,2024-08-23 00:00:00.000000,100016,Lumpsum,593268,Maharashtra,Mumbai,T30,26-35,Male,14.7,Net Banking,Pending
4,INV004096,2025-01-19 00:00:00.000000,120506,Lumpsum,591658,Tamil Nadu,Coimbatore,B30,36-45,Male,57.1,UPI,Pending
5,INV003815,2024-01-09 00:00:00.000000,119598,Lumpsum,591598,Gujarat,Surat,T30,36-45,Male,38.7,UPI,Verified
6,INV004404,2025-03-10 00:00:00.000000,101207,Lumpsum,590794,Telangana,Hyderabad,T30,26-35,Male,22.1,Mandate,Verified
7,INV000072,2025-04-17 00:00:00.000000,119599,Redemption,590635,Gujarat,Ahmedabad,T30,18-25,Female,6.4,UPI,Verified
8,INV000580,2024-08-06 00:00:00.000000,100033,Redemption,587757,Uttar Pradesh,Agra,B30,56+,Male,31.8,Cheque,Pending
9,INV001232,2024-08-27 00:00:00.000000,149322,Lumpsum,587530,Uttar Pradesh,Agra,B30,46-55,Female,29.8,UPI,Verified


In [49]:
pd.read_sql("""
SELECT
kyc_status,
COUNT(*) AS count
FROM investor_transactions
GROUP BY kyc_status;
""", engine)

,kyc_status,count
0,Pending,2632
1,Verified,30146


In [53]:
import pandas as pd

# Ensure 'scheme_performance' table is created and populated in the database
# using the 'perf' DataFrame, which is available in the kernel.
# This makes the cell self-contained and robust against state loss.
perf.to_sql(
    'scheme_performance',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql("""
SELECT
AVG(return_1yr_pct) AS avg_1y,
AVG(return_3yr_pct) AS avg_3y,
AVG(return_5yr_pct) AS avg_5y
FROM scheme_performance;
""", engine)

,avg_1y,avg_3y,avg_5y
0,14.376,14.089,14.51675


In [55]:
import pandas as pd

# Ensure 'scheme_performance' table is created and populated in the database
# using the 'perf' DataFrame, which is available in the kernel.
# This makes the cell self-contained and robust against state loss.
perf.to_sql(
    'scheme_performance',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql("""
SELECT *
FROM scheme_performance
ORDER BY return_1yr_pct DESC
LIMIT 10;
""", engine)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,101207,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Regular,24.93,22.38,23.80,20.54,1.84,0.97,0.90,1.47,25.0,-23.61,41613,1.53,5,Very High
1,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
2,119095,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Small Cap,Regular,21.97,20.98,22.62,20.47,0.51,1.00,0.84,1.40,25.0,-14.45,21545,1.38,4,Very High
3,118634,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,Regular,21.30,20.15,21.88,19.35,0.80,1.03,0.81,1.14,25.0,-30.87,43630,1.53,4,Very High
4,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,149324,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Small Cap,Regular,20.20,20.08,20.61,19.39,0.69,0.98,0.80,1.23,25.0,-17.01,35124,1.52,4,Very High
6,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High
7,102887,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,Flexi Cap,Regular,17.43,15.34,15.78,13.55,1.79,1.00,0.96,1.37,16.0,-12.14,17912,1.64,5,Moderately High
8,120842,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Mid Cap,Regular,17.12,18.23,17.75,16.32,1.91,1.00,0.96,1.27,19.0,-21.92,47469,1.56,4,High
9,120506,ICICI Pru Value Discovery Fund - Regular - Growth,ICICI Prudential MF,Value,Regular,16.67,14.76,12.60,14.21,0.55,0.92,0.98,1.50,15.0,-21.89,2571,1.41,4,Moderately High


In [57]:
import pandas as pd

# Ensure 'scheme_performance' table is created and populated in the database
# using the 'perf' DataFrame, which is available in the kernel.
# This makes the cell self-contained and robust against state loss.
perf.to_sql(
    'scheme_performance',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql("""
SELECT *
FROM scheme_performance
ORDER BY expense_ratio_pct ASC
LIMIT 10;
""", engine)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,118636,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Gilt,Regular,6.19,5.31,8.71,4.42,0.89,0.37,1.33,2.38,4.0,-2.23,30030,0.55,4,Low
1,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
2,120844,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Liquid,Regular,4.26,6.18,8.26,4.66,1.52,0.47,6.18,9.70,0.5,-3.81,27623,0.60,3,Low
3,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
4,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,118633,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Large Cap,Direct,14.42,12.33,14.72,10.63,1.70,1.02,0.88,1.48,14.0,-18.14,39475,0.72,3,Moderate
6,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
7,119093,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,Large Cap,Direct,13.95,12.14,13.66,10.71,1.43,0.87,0.87,1.54,14.0,-17.40,15866,0.75,4,Moderate
8,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
9,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High


In [59]:
import pandas as pd

# Ensure 'scheme_performance' table is created and populated in the database
# using the 'perf' DataFrame, which is available in the kernel.
# This makes the cell self-contained and robust against state loss.
perf.to_sql(
    'scheme_performance',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql("""
SELECT *
FROM scheme_performance
WHERE expense_ratio_pct < 1;
""", engine)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
1,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
2,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
3,125497,HDFC Top 100 Fund - Direct Plan - Growth,HDFC Mutual Fund,Large Cap,Direct,11.48,13.38,13.48,12.25,1.13,0.97,0.96,1.45,14.0,-33.50,10611,0.92,4,Moderate
4,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High
5,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
6,120504,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,Large Cap,Direct,14.12,14.41,13.02,13.53,0.88,1.03,1.03,1.27,14.0,-26.59,41553,0.80,3,Moderate
7,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
8,118633,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Large Cap,Direct,14.42,12.33,14.72,10.63,1.70,1.02,0.88,1.48,14.0,-18.14,39475,0.72,3,Moderate
9,118635,Nippon India ETF Nifty 50 BeES,Nippon India MF,Index/ETF,Direct,10.14,11.77,12.31,9.97,1.80,1.04,0.91,1.24,13.0,-26.75,20284,0.89,5,Moderate


In [61]:
pd.read_sql("""
SELECT
strftime('%Y-%m', date) AS month,
AVG(nav) AS avg_nav
FROM nav_history
GROUP BY month
ORDER BY month;
""", engine)

,month,avg_nav
0,2022-01,207.061394
1,2022-02,207.717759
2,2022-03,209.692614
3,2022-04,211.833457
4,2022-05,212.731451
5,2022-06,213.860867
6,2022-07,213.956111
7,2022-08,215.683994
8,2022-09,218.494307
9,2022-10,219.529633


In [63]:
import pandas as pd

# Ensure 'investor_transactions' table is created and populated in the database
# using the 'txn' DataFrame, which is available in the kernel.
# This makes the cell self-contained and robust against state loss.
txn.to_sql(
    'investor_transactions',
    engine,
    if_exists='replace',
    index=False
)

pd.read_sql("""
SELECT
strftime('%Y-%m', transaction_date) AS month,
SUM(amount_inr) AS total_amount
FROM investor_transactions
GROUP BY month
ORDER BY month;
""", engine)

,month,total_amount
0,2024-01,217648305
1,2024-02,193889416
2,2024-03,213453498
3,2024-04,208503874
4,2024-05,205126352
5,2024-06,222265329
6,2024-07,197732026
7,2024-08,227805125
8,2024-09,188561816
9,2024-10,213651354


In [64]:
pd.read_sql("""
SELECT *
FROM investor_transactions
WHERE transaction_type='SIP';
""", engine)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01 00:00:00.000000,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV003420,2024-01-01 00:00:00.000000,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
2,INV003436,2024-01-01 00:00:00.000000,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
3,INV001497,2024-01-01 00:00:00.000000,101208,SIP,3295,Maharashtra,Mumbai,T30,36-45,Male,56.8,Mandate,Verified
4,INV000786,2024-01-01 00:00:00.000000,101208,SIP,15047,Madhya Pradesh,Bhopal,B30,26-35,Male,17.9,Mandate,Verified
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19711,INV002074,2025-05-30 00:00:00.000000,118633,SIP,2383,Karnataka,Bangalore,T30,26-35,Female,6.1,UPI,Verified
19712,INV000339,2025-05-30 00:00:00.000000,148569,SIP,1109,Uttar Pradesh,Kanpur,B30,26-35,Male,22.3,Cheque,Verified
19713,INV001838,2025-05-30 00:00:00.000000,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified
19714,INV000074,2025-05-30 00:00:00.000000,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified


In [65]:
pd.read_sql("""
SELECT *
FROM investor_transactions
WHERE transaction_type='Redemption';
""", engine)

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV002952,2024-01-01 00:00:00.000000,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
1,INV000368,2024-01-01 00:00:00.000000,118636,Redemption,339882,Uttar Pradesh,Agra,B30,26-35,Male,16.2,UPI,Verified
2,INV004918,2024-01-01 00:00:00.000000,120503,Redemption,153828,Punjab,Amritsar,B30,56+,Female,46.9,UPI,Verified
3,INV002984,2024-01-01 00:00:00.000000,120844,Redemption,230179,Delhi,Gurugram,T30,26-35,Male,18.1,Mandate,Verified
4,INV000097,2024-01-01 00:00:00.000000,125498,Redemption,527377,Karnataka,Mysore,B30,18-25,Male,6.0,Net Banking,Verified
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4962,INV003258,2025-05-30 00:00:00.000000,148568,Redemption,19003,Rajasthan,Udaipur,B30,26-35,Male,14.7,UPI,Verified
4963,INV000651,2025-05-30 00:00:00.000000,118632,Redemption,69695,Delhi,New Delhi,T30,18-25,Male,6.0,Cheque,Verified
4964,INV002663,2025-05-30 00:00:00.000000,149323,Redemption,279886,Rajasthan,Udaipur,B30,36-45,Male,46.0,Net Banking,Verified
4965,INV000327,2025-05-30 00:00:00.000000,149323,Redemption,116572,Uttar Pradesh,Kanpur,B30,26-35,Male,9.2,Net Banking,Verified


In [67]:
!git add .
!git commit -m "Day 2: Cleaned data + SQLite DB loaded"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
